# Testing the `AdvisorWorkFlow _v1.3` n8n Workflow

Test cases are loaded from `unit_tests/advisor/*/test.json` (generated by the
unit-test generator).  Each file contains a pre-built `input` dict that is
posted verbatim to the webhook, and an `expectedOutput` dict used for
pass/fail comparison.

### What the webhook returns
```json
{
  "consultAcc": "...", "maxPayment": 0, "maxTerm": 0,
  "DebtSituation": "...", "refPlanID": null,
  "maxPaymentY2": null, "maxPaymentY3": null, "reConfirmMessage": ""
}
```

### URL note
- Test URL (n8n editor open): `{base}/webhook-test/{path}`
- Production URL (workflow Active): `{base}/webhook/{path}`

In [1]:
import json
from pathlib import Path

import pandas as pd
import requests

## 1. Configuration

In [2]:
N8N_BASE_URL = "https://alphamakeathon-automation.arisetech.dev"  # <-- change me
WEBHOOK_PATH = "b7607735-bac3-4965-8c68-2ed0ef38aec4"
USE_TEST_URL = False  # True -> use the "Listen for test event" URL instead


def get_webhook_url() -> str:
    prefix = "webhook-test" if USE_TEST_URL else "webhook"
    return f"{N8N_BASE_URL}/{prefix}/{WEBHOOK_PATH}"


get_webhook_url()

'https://alphamakeathon-automation.arisetech.dev/webhook/b7607735-bac3-4965-8c68-2ed0ef38aec4'

## 2. Load test cases from JSON

In [3]:
def _find_project_root() -> Path:
    p = Path.cwd()
    while p != p.parent:
        if (p / "unit_tests").is_dir():
            return p
        p = p.parent
    raise RuntimeError("Could not find project root (directory containing 'unit_tests/')")


def load_unit_tests(agent_type: str) -> list[dict]:
    root = _find_project_root()
    test_dir = root / "unit_tests" / agent_type
    tests = []
    for tc_dir in sorted(test_dir.iterdir()):
        test_file = tc_dir / "test.json"
        if test_file.is_file():
            with open(test_file, encoding="utf-8") as f:
                tests.append(json.load(f))
    print(f"Loaded {len(tests)} test cases from {test_dir}")
    return tests


TEST_CASES = load_unit_tests("advisor")
for tc in TEST_CASES:
    print(f"  {tc['testId']}: {tc['testDescription']}")

Loaded 5 test cases from /Users/tanut/Desktop/work/final_test_case_gen/unit_tests/advisor
  TC-0001: [strict] OFF_TOPIC — advisor extracts: consultAcc='', maxPayment=None
  TC-0002: [strict] MULTI_TURN — advisor extracts: consultAcc='', maxPayment=0
  TC-0003: [strict] OFF_TOPIC — advisor extracts: consultAcc='', maxPayment=None
  TC-0004: [adversarial] NEW_CONVERSATION — advisor extracts: consultAcc='', maxPayment=None
  TC-0005: [adversarial] AFTER_OFFER_TEXT — advisor extracts: consultAcc='1xxxxxxx0443,1xxxxxxx0442,1xxxxxxx0441', maxPayment=800


## 3. Webhook caller

In [4]:
def call_advisor(payload: dict, timeout: int = 60) -> dict:
    """POST the pre-built input dict to the Advisor webhook and return parsed JSON."""
    url = get_webhook_url()
    response = requests.post(url, json=payload, timeout=timeout)
    response.raise_for_status()
    return response.json()

## 4. Run a single example

Inspect one test case and confirm connectivity before running the full suite.

In [5]:
sample_tc = TEST_CASES[0]
print(f"Test: {sample_tc['testId']} — {sample_tc['testDescription']}")
print("\nInput payload:")
print(json.dumps(sample_tc["input"], ensure_ascii=False, indent=2))
print("\nExpected output:")
print(json.dumps(sample_tc["expectedOutput"], ensure_ascii=False, indent=2))

result = call_advisor(sample_tc["input"])
print("\nActual output:")
print(json.dumps(result, ensure_ascii=False, indent=2))

Test: TC-0001 — [strict] OFF_TOPIC — advisor extracts: consultAcc='', maxPayment=None

Input payload:
{
  "consultAcc": "",
  "DebtSituation": null,
  "maxPayment": 0,
  "maxTerm": 360,
  "refPlanID": null,
  "maxPaymentY2": null,
  "maxPaymentY3": null,
  "narrative": "",
  "accText": "[{\"account_number\": \"1xxxxxxx1459\", \"port\": \"สินเชื่อกรุงไทยใจป้ำ\", \"creditlimit\": 25000.0, \"os\": 18713.55, \"installment\": 1500.0, \"remaining_term\": 15}]",
  "offerText": "[]",
  "accInfotoExtr": [
    {
      "account_number": "1xxxxxxx1459",
      "port": "สินเชื่อกรุงไทยใจป้ำ",
      "creditlimit": 25000.0,
      "os": 18713.55,
      "installment": 1500.0,
      "remaining_term": 15
    }
  ],
  "LastAImessage": "\"สวัสดีครับ ยินดีให้บริการ คุณมีคำถามเกี่ยวกับการปรับโครงสร้างหนี้ไหมครับ\"",
  "userMessage": "สวัสดีค่ะ วันนี้อากาศดีไหมคะ"
}

Expected output:
{
  "consultAcc": "",
  "DebtSituation": null,
  "maxPayment": null,
  "maxTerm": 360,
  "refPlanID": null,
  "maxPaymentY2": nu

## 5. Run all test cases and compare against expectedOutput

In [6]:
COMPARE_KEYS = [
    "consultAcc", "DebtSituation", "maxPayment", "maxTerm",
    "refPlanID", "maxPaymentY2", "maxPaymentY3", "reConfirmMessage",
]

rows = []
for tc in TEST_CASES:
    try:
        result = call_advisor(tc["input"])
        expected = tc["expectedOutput"]
        mismatches = [k for k in COMPARE_KEYS if expected.get(k) != result.get(k)]
        rows.append({
            "testId": tc["testId"],
            "scenario": tc["scenario"],
            "messageMode": tc["messageMode"],
            "pass": len(mismatches) == 0,
            "mismatched_fields": ", ".join(mismatches) if mismatches else "",
            **{f"exp_{k}": expected.get(k) for k in COMPARE_KEYS},
            **{f"act_{k}": result.get(k) for k in COMPARE_KEYS},
            "error": None,
        })
    except requests.exceptions.RequestException as exc:
        rows.append({
            "testId": tc["testId"],
            "scenario": tc["scenario"],
            "messageMode": tc["messageMode"],
            "pass": False,
            "mismatched_fields": "",
            **{f"exp_{k}": None for k in COMPARE_KEYS},
            **{f"act_{k}": None for k in COMPARE_KEYS},
            "error": str(exc),
        })

results_df = pd.DataFrame(rows)
summary_cols = ["testId", "scenario", "messageMode", "pass", "mismatched_fields", "error"]
print(f"Passed: {results_df['pass'].sum()}/{len(results_df)}")
results_df[summary_cols]

Passed: 2/5


,testId,scenario,messageMode,pass,mismatched_fields,error
0,TC-0001,OFF_TOPIC,strict,False,maxPayment,None
1,TC-0002,MULTI_TURN,strict,True,,None
2,TC-0003,OFF_TOPIC,strict,False,"maxPayment, maxTerm",None
3,TC-0004,NEW_CONVERSATION,adversarial,False,"DebtSituation, maxPayment",None
4,TC-0005,AFTER_OFFER_TEXT,adversarial,True,,None


## Notes

- This workflow is **Active**, so the production URL is
  `{base_url}/webhook/b7607735-bac3-4965-8c68-2ed0ef38aec4`.
- The webhook only returns the raw extractor output — to test the full
  advisor pipeline (offer-engine, clarify reply), invoke as a sub-workflow.
- Add more test cases by running the unit-test generator; they are picked up
  automatically the next time this notebook runs.
- If the n8n webhook requires authentication, add `auth=` or `headers=` to
  `requests.post` inside `call_advisor`.